In [ ]:
!curl -sL -O https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [ ]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

## Data Preparation

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum()

## Q1
What is the most frequent observation (mode) for the column `industry`?

In [ ]:
df.industry.value_counts()

In [ ]:
df.industry.mode()[0]

Answer: **`technology`**

## Q2
Create the correlation matrix for the numerical features of your dataset. What are the two features that have the biggest correlation?

In [ ]:
df[numerical].corr()

Answer: **`interaction_count` and `lead_score`** (`0.916`)

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)

## Q3
Calculate the mutual information score between `converted` and other categorical variables in the dataset using the training set only. Which of these variables has the biggest mutual information score?

In [ ]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False).round(2)

Answer: **`lead_source`** (`0.03`)

## Q4
Train a logistic regression model with one-hot encoding (`solver='liblinear', C=1.0, max_iter=1000, random_state=42`) and calculate the accuracy on the validation dataset rounded to 2 decimal digits.

In [ ]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
orig_score = (y_val == converted_decision).mean()
orig_score, round(float(orig_score), 2)

Answer: **`0.65`** (unrounded validation accuracy: `0.645`)

## Q5
Find the least useful feature using the feature elimination technique. Which of `'lead_source'`, `'number_of_courses_viewed'`, `'interaction_count'` has the smallest difference?

In [ ]:
features = categorical + numerical

for f in features:
    small = [c for c in features if c != f]

    dv_small = DictVectorizer(sparse=False)
    train_dict_small = df_train[small].to_dict(orient='records')
    X_train_small = dv_small.fit_transform(train_dict_small)

    val_dict_small = df_val[small].to_dict(orient='records')
    X_val_small = dv_small.transform(val_dict_small)

    model_small = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_small.fit(X_train_small, y_train)

    y_pred_small = model_small.predict_proba(X_val_small)[:, 1]
    score = (y_val == (y_pred_small >= 0.5)).mean()
    diff = orig_score - score
    print(f'{f:25s}: accuracy={score:.4f}, diff={diff:+.4f}, abs_diff={abs(diff):.4f}')

Answer: **`'number_of_courses_viewed'`** (accuracy `0.6430`, difference `0.0020`)

## Q6
Train a regularized logistic regression with `C` in `[0.000001, 0.00001, 0.0001, 0.001]`. Which of these `C` leads to the best accuracy on the validation set?

In [ ]:
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model_reg = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_reg.fit(X_train, y_train)

    y_pred_reg = model_reg.predict_proba(X_val)[:, 1]
    score = (y_val == (y_pred_reg >= 0.5)).mean()
    print(f'C={c:<8}: {round(score, 3)}')

Answer: **`0.001`** (`0.645`)